In [1]:
# Can this be done for GMM? Read colored noise thermostats.
import numpy as np
from scipy.optimize import minimize
from matrix_exp import stationary_covariance, compute_mean_and_covariance

# === hyperparameters ===
M = 1.0
beta = 8*np.sqrt(M)
T = 1.0
# === Range of omega frequencies ====
omega_min = 0.1
omega_max = 10.
n_omega = 1000
omegas = np.linspace(omega_min, omega_max, n_omega)

def build_matrices(gamma, lamb, c):
    # F = -beta * A
    A = np.array([[0.0, -1.0/M, 0.0],
                  [1.0, (1.0/M)*(gamma**2), gamma*lamb*c],
                  [0.0, gamma*lamb*c, lamb**2]])
    # G = sqrt(2 * beta) * B
    B = np.array([[0.0, 0.0, 0.0],
                  [0.0, gamma, 0.0],
                  [0.0, lamb*c, lamb*np.sqrt(1.0-c**2)]])
    return A, B

def objective_analytic(params):
    gamma, lamb, c = params
    if gamma <= 0 or lamb <= 0 or not (0.0 < c < 1.0):
        return 1e6

    # Build system matrices
    A, B = build_matrices(gamma, lamb, c)
    G = np.sqrt(2*beta) * B

    # Stationary covariance by solving Lyapunov Equation: FC + CF^T - GG^T = 0
    C = stationary_covariance(beta, A, G)

    total = 0.0
    for w in omegas:
        Sigma0 = np.diag([w**(-2), M, 1.0])
        mu_t, SigmaT = compute_mean_and_covariance(
            T, beta, A, G, mu_0=np.zeros(3), Sigma_0=Sigma0, C=C
        )
        sigma_x_sq = max(1e-16, SigmaT[0,0])
        sigma_x = np.sqrt(sigma_x_sq)
        r = w * sigma_x
        kl = np.log(r) + r**(-2) - 0.5
        total += kl

    avg = total / len(omegas)
    interval = omegas[-1] - omegas[0]
    return avg * interval

# === run optimization ===
x0 = np.array([1.0, 1.0, 0.5])
bounds = [(0., 8.0), (0., 8.0), (1e-6, 1-1e-6)]
res = minimize(lambda x: objective_analytic(x), x0, method='L-BFGS-B', bounds=bounds,
               options={'maxiter': 200})
print("Success:", res.success, res.message)
print("Best (gamma, lambda, c) =", res.x, " objective =", res.fun)



/disk/homeDIRS/rrajpal/GeneralizedLangevinDiffusion/gld/gmm/matrix_exp.py:9: RuntimeWarning: Input "a" has an eigenvalue pair whose sum is very close to or exactly zero. The solution is obtained via perturbing the coefficients.
  return solve_continuous_lyapunov(F, -Q)


Success: True CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Best (gamma, lambda, c) = [6.30351027 8.         0.999999  ]  objective = 4.008216611941872
